In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
spark = SparkSession.builder \
    .appName("Smart Education Analytics") \
    .getOrCreate()

df = spark.read.csv(
    "smart_education_single_dirty_dataset.csv",
    header=True,
    inferSchema=True
)

df.show(5)
df.printSchema()
print("Total Rows:", df.count())
print("Total Columns:", len(df.columns))

In [ ]:
rdd = df.rdd

print("First 5 RDD Records:")
for row in rdd.take(5):
    print(row)

dept_perf_rdd = rdd.map(lambda row: (row["department"], row["performance_label"]))

print("Department Performance Sample:")
print(dept_perf_rdd.take(10))
print("RDD Count:", rdd.count())

In [ ]:
placed_rdd = rdd.filter(lambda row: row["placement_status"] == "Placed")

print("Placed Students Count:", placed_rdd.count())

for row in placed_rdd.take(5):
    print(row["student_id"], row["department"], row["placement_status"])

In [ ]:
# Key-value pair: department wise student count
dept_count_rdd = rdd.map(lambda row: (row["department"], 1)) \
                    .reduceByKey(lambda a, b: a + b)

print("Department-wise Student Count:")
print(dept_count_rdd.collect())

In [ ]:
cached_rdd = rdd.cache()

print("Cached RDD Count:", cached_rdd.count())

In [ ]:
df_clean = df.dropDuplicates()

df_clean = df_clean.fillna({
    "gender": "Unknown",
    "city": "Unknown",
    "family_income_group": "Unknown",
    "attendance_percentage": 0,
    "hours_studied_online": 0,
    "videos_watched": 0,
    "internal_marks": 0,
    "final_exam_marks": 0,
    "technical_skill_score": 0,
    "aptitude_score": 0,
    "communication_score": 0,
    "interview_score": 0
})

df_clean.show(5)
print("After Cleaning Rows:", df_clean.count())

In [ ]:
df_clean = df_clean.filter(
    (col("age").between(18, 30)) &
    (col("attendance_percentage").between(0, 100)) &
    (col("hours_studied_online").between(0, 250)) &
    (col("package_lpa").between(0, 25))
)

print("Rows After Outlier Removal:", df_clean.count())

In [ ]:
dept_summary = df_clean.groupBy("department").agg(
    count("student_id").alias("total_students"),
    round(avg("attendance_percentage"), 2).alias("avg_attendance"),
    round(avg("cgpa"), 2).alias("avg_cgpa"),
    round(avg("package_lpa"), 2).alias("avg_package")
)

dept_summary.show()

In [ ]:
student_df = df_clean.select(
    "student_id", "student_name", "department", "semester"
).dropDuplicates()

academic_df = df_clean.select(
    "student_id", "cgpa", "total_marks", "performance_label"
).dropDuplicates()

joined_df = student_df.join(
    academic_df,
    on="student_id",
    how="inner"
)

joined_df.show(10)

In [ ]:
df_clean.createOrReplaceTempView("education")

In [ ]:
semester_report = spark.sql("""
SELECT
    semester,
    ROUND(AVG(cgpa), 2) AS avg_cgpa,
    ROUND(AVG(total_marks), 2) AS avg_marks,
    ROUND(AVG(attendance_percentage), 2) AS avg_attendance,
    COUNT(*) AS total_students
FROM education
GROUP BY semester
ORDER BY semester
""")

semester_report.show()

In [ ]:
attendance_sql = spark.sql("""
SELECT
    department,
    ROUND(AVG(attendance_percentage), 2) AS avg_attendance
FROM education
GROUP BY department
ORDER BY avg_attendance DESC
""")

attendance_sql.show()

In [ ]:
subject_performance = spark.sql("""
SELECT
    subject,
    ROUND(AVG(total_marks), 2) AS avg_marks,
    COUNT(*) AS total_students
FROM education
GROUP BY subject
ORDER BY avg_marks DESC
""")

subject_performance.show()

In [ ]:
top_students = spark.sql("""
SELECT
    student_id,
    student_name,
    department,
    cgpa,
    performance_label
FROM education
ORDER BY cgpa DESC
LIMIT 10
""")

top_students.show()

In [ ]:
placement_trends = spark.sql("""
SELECT
    department,
    placement_status,
    COUNT(*) AS total
FROM education
GROUP BY department, placement_status
ORDER BY department
""")

placement_trends.show()

In [ ]:
def etl_pipeline(input_df):
    cleaned = input_df.dropDuplicates()

    cleaned = cleaned.fillna({
        "gender": "Unknown",
        "city": "Unknown",
        "family_income_group": "Unknown",
        "attendance_percentage": 0,
        "hours_studied_online": 0,
        "videos_watched": 0,
        "internal_marks": 0,
        "final_exam_marks": 0,
        "technical_skill_score": 0,
        "aptitude_score": 0,
        "communication_score": 0,
        "interview_score": 0
    })

    cleaned = cleaned.filter(
        (col("age").between(18, 30)) &
        (col("attendance_percentage").between(0, 100)) &
        (col("hours_studied_online").between(0, 250)) &
        (col("package_lpa").between(0, 25))
    )

    return cleaned

final_df = etl_pipeline(df)

final_df.show(5)
print("Final ETL Rows:", final_df.count())

In [ ]:
final_df.write.mode("overwrite").csv("processed_smart_education_data", header=True)

In [ ]:
internship_indexer = StringIndexer(
    inputCol="internship_completed",
    outputCol="internship_index"
)

label_indexer = StringIndexer(
    inputCol="placement_status",
    outputCol="label"
)

feature_cols = [
    "attendance_percentage",
    "login_count",
    "hours_studied_online",
    "assignments_submitted",
    "quiz_attempts",
    "cgpa",
    "technical_skill_score",
    "aptitude_score",
    "communication_score",
    "interview_score",
    "internship_index"
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=50,
    maxDepth=5,
    seed=42
)

pipeline = Pipeline(stages=[
    internship_indexer,
    label_indexer,
    assembler,
    rf
])

In [ ]:
ml_df = final_df.select(
    "attendance_percentage",
    "login_count",
    "hours_studied_online",
    "assignments_submitted",
    "quiz_attempts",
    "cgpa",
    "technical_skill_score",
    "aptitude_score",
    "communication_score",
    "interview_score",
    "internship_completed",
    "placement_status"
).dropna()

ml_df.show(5)

In [ ]:
train_data, test_data = ml_df.randomSplit([0.8, 0.2], seed=42)

model = pipeline.fit(train_data)

predictions = model.transform(test_data)

predictions.select(
    "placement_status",
    "label",
    "prediction",
    "probability"
).show(20, truncate=False)

In [ ]:
accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

accuracy = accuracy_evaluator.evaluate(predictions)
f1_score = f1_evaluator.evaluate(predictions)

print("Model Accuracy:", accuracy)
print("F1 Score:", f1_score)

In [ ]:
model.write().overwrite().save("student_placement_prediction_model")